# AmsiNet - Step 13: 12-way script identification, matched to the BSTD (IJDAR) protocol

**Runtime:** Colab **A100** (Parts A and B train; Part C is CPU work in the same session). Needs Step 2 outputs: `crops224.tar`, `crops_meta.csv` (with the `lang` column) and `bstd_index.json`.

**Why.** The BSTD paper reports 12-way accuracy of 80.5% (ViT-B) and 67.7% (CLIP) on a class-balanced split: 1,800 training and 478 test words per language, 12 labels with Hindi/Marathi and Assamese/Bengali **not** merged. Our ten-class protocol merges those pairs, so the numbers are not comparable. This notebook reruns **our** models in their label space.

* **Part A (their protocol):** flat ViT-B/16, 12 labels, trained on a random draw of 1,800 words per language from our training images; evaluated on a random draw of 478 words per language from our test images (a fresh draw per seed). This is the closest we can get to their protocol without their `script_identification.zip` split (if you have that file, tell Claude and we will use it instead).
* **Part B (full split):** flat ViT-B/16 and the EfficientViT-B0 word agent, 12 labels, trained on all our training words.
* **Part C (CPU):** accuracy and macro-F1 on (i) the balanced 478-per-language test draw and (ii) all test words, mean +/- 95% t-interval over seeds; AmsiNet's **free word context** (leave-one-out line/block context, weights fitted on validation) applied to the 12-way word agent; and how many errors are the merged pairs (Hindi-Marathi, Assamese-Bengali). Line and block agents are not retrained for 12 labels here; free context already gave 87% of the gain in the ten-class study.

In [ ]:
# 1. Setup
from google.colab import drive
import os, subprocess, sys
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
try: import timm
except ImportError: subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'timm'], check=True); import timm
import json, glob, random, time, math, numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from scipy import stats
from scipy.special import softmax
from sklearn.metrics import f1_score
ROOT, LOCAL = '/content/drive/MyDrive/amsinet', '/content/bstd'; os.makedirs(f'{ROOT}/agents12', exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
CFG = dict(SEEDS=[0, 1, 2], N_TRAIN_PER_LANG=1800, N_TEST_PER_LANG=478, N_VAL_PER_LANG=200,
           FLAT_BAL_EPOCHS=10, FLAT_FULL_EPOCHS=5, WORD_EPOCHS=8, BATCH=64, WORKERS=8, EPS=1e-4)
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8')); SCRIPTS, INDEX = D['scripts'], D['index']; IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); assert 'lang' in META.columns, 'crops_meta.csv has no lang column'
LANGS = sorted(META.lang.dropna().unique()); LAB = {l: i for i, l in enumerate(LANGS)}; C12 = len(LANGS); META['y12'] = META.lang.map(LAB).astype(int)
print(C12, 'language labels:', LANGS); assert C12 == 12, 'expected 12 languages: check the lang tags above and tell Claude'
print(META.groupby(['split', 'lang']).size().unstack(0))
CROPS, CROPS_TAR = f'{LOCAL}/crops224', f'{ROOT}/data/crops224.tar'
if not os.path.isdir(CROPS) and os.path.exists(CROPS_TAR): print('restoring crops'); subprocess.run(['tar', '-xf', CROPS_TAR, '-C', LOCAL], check=True)
assert os.path.exists(META.path.iloc[0]), 'crops not found on disk: ' + META.path.iloc[0]
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## 2. Training utilities

In [ ]:
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
DEV = 'cuda'; FILL = (124, 116, 104)
class Crops(Dataset):
    def __init__(self, paths, labels, tf): self.p, self.y, self.tf = list(paths), list(labels), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert('RGB')), self.y[i]
def tfs(model):
    cfg = timm.data.resolve_data_config({}, model=model); norm = T.Normalize(cfg['mean'], cfg['std'])
    return (T.Compose([T.RandomAffine(5, translate=(0.03, 0.03), scale=(0.9, 1.1), fill=FILL), T.ColorJitter(0.3, 0.3, 0.2), T.ToTensor(), norm]), T.Compose([T.ToTensor(), norm]))
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
@torch.no_grad()
def predict(model, df, tf):
    model.eval(); out = []
    for x, _ in DataLoader(Crops(df.path, df.y12, tf), batch_size=256, num_workers=CFG['WORKERS']):
        with torch.autocast('cuda', dtype=torch.float16): out.append(model(x.to(DEV)).float().softmax(1).cpu())
    return torch.cat(out).numpy()
def mf1(P, y): return float(f1_score(y, P.argmax(1), average='macro', labels=list(range(C12)), zero_division=0))
def train(tag, model_name, tr, va, epochs, lr, seed):
    # trains on dataframe tr, selects the epoch by macro-F1 on dataframe va, saves probabilities for ALL val and test words
    files = [f'{ROOT}/agents12/{tag}_seed{seed}_{sp}_probs.npy' for sp in ('val', 'test')]
    if all(os.path.exists(f) for f in files): print('skip (done):', tag, seed); return
    set_seed(seed); model = timm.create_model(model_name, pretrained=True, num_classes=C12).to(DEV); ttf, etf = tfs(model)
    L = DataLoader(Crops(tr.path, tr.y12, ttf), batch_size=CFG['BATCH'], shuffle=True, num_workers=CFG['WORKERS'], pin_memory=True, drop_last=True)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    total = epochs * len(L); warm = max(1, int(0.05 * total)); sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler('cuda'); best, state = -1, None
    for ep in range(epochs):
        model.train(); t0 = time.time(); rl = 0
        for x, y in L:
            x, y = x.to(DEV, non_blocking=True), torch.as_tensor(y).to(DEV)
            with torch.autocast('cuda', dtype=torch.float16): loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step(); rl += loss.item()
        Pv = predict(model, va, etf); f = mf1(Pv, va.y12.values)
        print(f'{tag} seed{seed} ep{ep + 1}: loss {rl / len(L):.4f} val acc {(Pv.argmax(1) == va.y12.values).mean():.4f} macro-F1 {f:.4f} ({time.time() - t0:.0f}s)')
        if f > best: best = f; state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(state)
    for sp in ('val', 'test'):
        d = META[META.split == sp]; P = predict(model, d, etf); np.save(f'{ROOT}/agents12/{tag}_seed{seed}_{sp}_probs.npy', P.astype(np.float16))
    print(f'done {tag} seed{seed}: best val macro-F1 {best:.4f}'); del model; torch.cuda.empty_cache()
def draw(split, n, seed):
    rs = np.random.RandomState(1000 + seed); d = META[META.split == split]; idx = []
    for l in LANGS:
        ii = d.index[d.lang == l].values; idx += list(rs.choice(ii, min(n, len(ii)), replace=False))
    return META.loc[sorted(idx)]

## Part A: the BSTD-style balanced protocol (flat ViT-B/16, 1,800 training words per language)

In [ ]:
for seed in CFG['SEEDS']:
    tr = draw('train', CFG['N_TRAIN_PER_LANG'], seed); va = draw('val', CFG['N_VAL_PER_LANG'], seed); print('seed', seed, 'train words', len(tr), '| val words', len(va))
    train('flat12bal', 'vit_base_patch16_224.augreg_in21k', tr, va, CFG['FLAT_BAL_EPOCHS'], 5e-5, seed)

## Part B: full training split (flat ViT-B/16 and the EfficientViT-B0 word agent)

In [ ]:
TRN, VAL = META[META.split == 'train'], META[META.split == 'val']
for seed in CFG['SEEDS']:
    train('flat12full', 'vit_base_patch16_224.augreg_in21k', TRN, VAL, CFG['FLAT_FULL_EPOCHS'], 5e-5, seed)
    train('word12', 'efficientvit_b0.r224_in1k', TRN, VAL, CFG['WORD_EPOCHS'], 1e-4, seed)
print('training done')

## Part C: evaluation (CPU)

In [ ]:
def load(tag, seed, sp): return np.load(f'{ROOT}/agents12/{tag}_seed{seed}_{sp}_probs.npy').astype(np.float32)
def acc_f1(P, y): return 100 * float((P.argmax(1) == y).mean()), 100 * mf1(P, y)
# free word context for the 12-way word agent (same construction as the ten-class study)
def groups(sp):
    d = META[META.split == sp]; lid, bid = [], []
    for iid, k in zip(d.image_id, d.word_idx):
        w = IDX[iid]['words'][k]; lid.append(w.get('line', -1)); bid.append(w.get('block', -1))
    img = d.image_id.values.astype(str); lg = pd.factorize(img + '|l' + np.array(lid).astype(str))[0]; bg = pd.factorize(img + '|b' + np.array(bid).astype(str))[0]; return lg, bg
GR = {sp: groups(sp) for sp in ('val', 'test')}
def loo(g, logP):
    n = len(g); ng = g.max() + 1; S = np.zeros((ng, C12)); cnt = np.zeros(ng); np.add.at(S, g, logP); np.add.at(cnt, g, 1); o = cnt[g] - 1
    F = np.zeros((n, C12)); m = o > 0; F[m] = (S[g][m] - logP[m]) / o[m][:, None]; return F
def ctx_scores(P, sp, a1, a2):
    logP = np.log(np.clip(P, CFG['EPS'], 1)); lg, bg = GR[sp]; return logP + a1 * loo(lg, logP) + a2 * loo(bg, logP)
GRID = [0, .1, .25, .5, 1, 1.5, 2, 3]
def fit_ctx(Pv, yv):
    a = {'a1': 0.0, 'a2': 0.0}
    for _ in range(2):
        for k in a: a[k] = max(GRID, key=lambda g: mf1(ctx_scores(Pv, 'val', **{**a, k: g}), yv))
    return a
yv_all = META[META.split == 'val'].y12.values; yt_all = META[META.split == 'test'].y12.values; TEST = META[META.split == 'test']
pos = {ix: i for i, ix in enumerate(TEST.index)}
L2S = META.drop_duplicates('lang').set_index('lang').script.to_dict(); PAIRS = [('hindi', 'marathi'), ('assamese', 'bengali')]
rows = []
for seed in CFG['SEEDS']:
    bal = draw('test', CFG['N_TEST_PER_LANG'], seed); bi = np.array([pos[i] for i in bal.index]); yb = yt_all[bi]
    P = {k: load(k, seed, 'test') for k in ('flat12bal', 'flat12full', 'word12')}; Pv = load('word12', seed, 'val'); a = fit_ctx(Pv, yv_all)
    P['AmsiNet free context (12-way word agent + context)'] = softmax(ctx_scores(P['word12'], 'test', **a), axis=1)
    for k, p in P.items():
        ab, fb = acc_f1(p[bi], yb); af, ff = acc_f1(p, yt_all); pr = p.argmax(1); err = pr != yt_all
        pair = 0
        for x, y_ in PAIRS: pair += (((yt_all == LAB[x]) & (pr == LAB[y_])) | ((yt_all == LAB[y_]) & (pr == LAB[x]))).sum() if x in LAB and y_ in LAB else 0
        s10 = np.mean([L2S[LANGS[i]] == L2S[LANGS[j]] or i == j for i, j in zip(yt_all[err], pr[err])]) if err.any() else 0
        rows.append(dict(seed=seed, system=k, bal_acc=ab, bal_f1=fb, full_acc=af, full_f1=ff, pair_share_of_errors=100 * pair / err.sum(), acc10=af + 100 * float(err.mean() * s10)))
R = pd.DataFrame(rows); R.to_csv(f'{ROOT}/results/twelve_way_per_seed.csv', index=False)
def mci(x):
    x = np.asarray(x, float); n = len(x); return x.mean(), (stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n) if n > 1 else float('nan'))
f = lambda x: '%.2f +/- %.2f' % mci(x)
NAMES = {'flat12bal': 'flat ViT-B/16, trained on 1,800 words per language (BSTD-style protocol)', 'flat12full': 'flat ViT-B/16, trained on the full training split', 'word12': 'word agent (EfficientViT-B0), full training split'}
T12 = pd.DataFrame([(NAMES.get(s, s), f(g.bal_acc), f(g.bal_f1), f(g.full_acc), f(g.full_f1), f(g.pair_share_of_errors)) for s, g in R.groupby('system', sort=False)],
                   columns=['system (12 labels)', 'balanced test: accuracy %', 'balanced test: macro-F1 %', 'full test: accuracy %', 'full test: macro-F1 %', '% of errors that are Hindi/Marathi or Assamese/Bengali'])
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 90); display(T12); T12.to_csv(f'{ROOT}/results/twelve_way_summary.csv', index=False)
print('Published in the BSTD paper (12-way, balanced test, their split): ViT-B 80.5%, CLIP 67.7% accuracy. Their split differs from ours (random draws from our training and test images).')